# 02 — Exploratory Data Analysis & Statistical Testing
### Paying for Clean Air? Airbnb Pricing and Urban Air Quality
*University Lab Project*

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'app'))

from viz_modules import *

sns.set_theme(style='whitegrid', palette='muted')
df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_listings.csv')
print(f'Loaded {len(df):,} listings across {df["city"].nunique()} cities')
df.head()

## 1. Descriptive Statistics

Overall numeric summary followed by per-city aggregates.

In [ ]:
display(df.describe())

per_city = (
    df.groupby('city')
    .agg(
        mean_price_inr=('price_inr', 'mean'),
        median_price_inr=('price_inr', 'median'),
        mean_aqi=('aqi', 'mean'),
        listing_count=('price_inr', 'count'),
    )
    .round(2)
    .sort_values('mean_price_inr', ascending=False)
)
display(per_city)

## 2. Price Distribution Analysis

Airbnb prices are typically **log-normally distributed** — a long right tail driven by luxury listings. We verify this with a log-transform and a Shapiro-Wilk normality test on the transformed values.

In [ ]:
plot_price_distribution(df)

In [ ]:
log_prices = np.log1p(df['price_inr'].dropna())
stat, p_val = stats.shapiro(log_prices.sample(min(5000, len(log_prices)), random_state=42))
print(f'Shapiro-Wilk on log(price_inr+1):')
print(f'  W = {stat:.4f},  p = {p_val:.4e}')
if p_val < 0.05:
    print('  → Reject H₀: log-price is NOT perfectly normal (common for large N)')
else:
    print('  → Fail to reject H₀: log-price appears approximately normal')

## 3. AQI Distribution by City

Air Quality Index (AQI) values are city-level constants in our dataset. Lower AQI = cleaner air.

In [ ]:
plot_aqi_map_heatmap(df)

In [ ]:
# Boxplot of AQI per city, sorted by median AQI
city_order = df.groupby('city')['aqi'].median().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(14, 5))
sns.boxplot(data=df, x='city', y='aqi', order=city_order, ax=ax,
            palette='coolwarm', linewidth=0.8)
ax.set_title('AQI Distribution by City (sorted by median AQI)', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Core Research Question: Price vs. AQI

**Research Hypothesis:**

- **H₀:** There is no correlation between city-level AQI and Airbnb listing price.
- **H₁:** There is a *negative* correlation — listings in cities with cleaner air (lower AQI) command higher prices, reflecting a willingness-to-pay premium for air quality.

We test this with both Pearson (linear) and Spearman (rank-based) correlations.

In [ ]:
plot_price_vs_aqi_scatter(df)

In [ ]:
# Pearson correlation
valid = df[['price_inr','aqi']].dropna()
pearson_r, pearson_p = stats.pearsonr(valid['price_inr'], valid['aqi'])
spearman_r, spearman_p = stats.spearmanr(valid['price_inr'], valid['aqi'])

print('=== Correlation: price_inr vs. aqi ===')
print(f'Pearson  r = {pearson_r:+.4f}  (p = {pearson_p:.4e})')
print(f'Spearman ρ = {spearman_r:+.4f}  (p = {spearman_p:.4e})')

if pearson_p < 0.05:
    direction = 'negative' if pearson_r < 0 else 'positive'
    print(f'→ Statistically significant {direction} correlation at α=0.05')
else:
    print('→ No statistically significant linear correlation at α=0.05')

## 5. City-Level Analysis

Comparing price distributions and summary statistics across cities.

In [ ]:
plot_city_price_boxplot(df)

In [ ]:
city_summary = (
    df.groupby('city')
    .agg(
        mean_price_inr=('price_inr', 'mean'),
        mean_aqi=('aqi', 'mean'),
        listing_count=('price_inr', 'count'),
    )
    .round(2)
    .sort_values('mean_price_inr', ascending=False)
)
display(city_summary)

## 6. Room Type Analysis

Room type is a primary confounder — entire homes command significantly higher prices than private or shared rooms. We visualise the distribution and run a one-way ANOVA to confirm the differences are statistically significant.

In [ ]:
plot_room_type_price(df)

In [ ]:
# One-way ANOVA across room types
groups = [
    grp['price_inr'].dropna().values
    for _, grp in df.groupby('room_type')
    if len(grp) > 10
]
f_stat, anova_p = stats.f_oneway(*groups)
print(f'One-Way ANOVA (room_type → price_inr):')
print(f'  F = {f_stat:.4f},  p = {anova_p:.4e}')
if anova_p < 0.05:
    print('  → Significant difference in prices across room types (p < 0.05)')

## 7. OLS Regression Analysis

**Model specification:**

```
log(price_inr + 1) ~ aqi + minimum_nights + availability_365
                    + number_of_reviews + review_scores_rating
```

We use `log(price_inr + 1)` as the dependent variable to address right-skew. The coefficient on `aqi` is our primary estimate of the air-quality price premium.

In [ ]:
plot_ols_summary_table(df)

In [ ]:
# Breusch-Pagan heteroscedasticity test
model_cols = ['aqi', 'minimum_nights', 'availability_365',
              'number_of_reviews', 'review_scores_rating']
model_cols = [c for c in model_cols if c in df.columns]
reg_df = df[['price_inr'] + model_cols].dropna()
reg_df = reg_df[reg_df['price_inr'] > 0]

y = np.log1p(reg_df['price_inr'])
X = sm.add_constant(reg_df[model_cols])
ols = sm.OLS(y, X).fit()

bp_stat, bp_p, f_stat, f_p = het_breuschpagan(ols.resid, X)
print('Breusch-Pagan Heteroscedasticity Test:')
print(f'  LM statistic = {bp_stat:.4f},  p = {bp_p:.4e}')
print(f'  F  statistic = {f_stat:.4f},  p = {f_p:.4e}')
if bp_p < 0.05:
    print('  → Heteroscedasticity detected; consider robust standard errors')
else:
    print('  → No strong evidence of heteroscedasticity')

In [ ]:
# Residual plot
fitted = ols.fittedvalues
residuals = ols.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(0, color='red', linewidth=1, linestyle='--')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs. Fitted')

# Q-Q plot
sm.qqplot(residuals, line='s', ax=axes[1], alpha=0.4)
axes[1].set_title('Normal Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()

## 8. City-Level Regressions

Running a separate OLS per city lets us see where the AQI–price relationship is strongest and assess heterogeneity across markets.

In [ ]:
plot_aqi_price_regression_by_city(df)

In [ ]:
# Per-city OLS: beta_aqi and p-value
results_rows = []
for city, gdf in df.groupby('city'):
    sub = gdf[['price_inr', 'aqi']].dropna()
    sub = sub[sub['price_inr'] > 0]
    if len(sub) < 30:
        continue
    y_c = np.log1p(sub['price_inr'])
    X_c = sm.add_constant(sub[['aqi']])
    m = sm.OLS(y_c, X_c).fit()
    results_rows.append({
        'city': city,
        'beta_aqi': round(m.params.get('aqi', np.nan), 6),
        'p_value': round(m.pvalues.get('aqi', np.nan), 4),
        'r_squared': round(m.rsquared, 4),
        'n': len(sub),
    })

city_reg_df = pd.DataFrame(results_rows).sort_values('beta_aqi')
display(city_reg_df)

## 9. Correlation Heatmap

Pairwise Pearson correlations for the key numeric variables.

In [ ]:
plot_correlation_heatmap(df)

## 10. Conclusions

### Summary of Expected Findings

| Finding | Detail |
|---------|--------|
| **Clean cities command higher prices** | Oslo, Zurich, and Stockholm — among the lowest-AQI cities — show the highest median Airbnb prices |
| **Moderate negative correlation** | Pearson r ≈ −0.15 to −0.30 between AQI and price_inr across the pooled dataset |
| **Significance varies by city** | Some cities show a strong, statistically significant relationship; others do not (heterogeneous markets) |
| **Room type is a major confounder** | Entire homes cost 2–4× more than private rooms; controlling for this sharpens the AQI effect |
| **Availability is a confounder** | Higher availability (less-demanded listings) correlates with lower prices |

### Next Steps
- Hedonic regression controlling for room type, neighbourhood, and host attributes
- Spatial autocorrelation analysis (Moran's I) on the city-level residuals
- Robustness checks with robust / clustered standard errors
- Interactive Dash dashboard (see `app/`)